<a href="https://colab.research.google.com/github/AnoushkaTripathi/USR_RTLtoGDSII/blob/main/digital-inverter-librelane.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Digital inverter with OpenLane

```
Copyright 2022 Google LLC.
SPDX-License-Identifier: Apache-2.0
```

Run a simple digital inverter design thru the [OpenLane](https://github.com/The-OpenROAD-Project/OpenLane/) GDS to RTL flow targeting the [open source SKY130 PDK](https://github.com/google/skywater-pdk/).

In [9]:
#@title Install LibreLane (latest, fixed) {display-mode: "form"}

import os
import pathlib

# Install micromamba
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba

# Create clean conda env
conda_prefix_path = pathlib.Path('conda-env')
CONDA_PREFIX = str(conda_prefix_path.resolve())
!bin/micromamba create --yes --prefix $CONDA_PREFIX python=3.10 git make

# Activate environment
PATH = os.environ['PATH']
%env CONDA_PREFIX={CONDA_PREFIX}
%env PATH={CONDA_PREFIX}/bin:{PATH}

# Upgrade pip
!python -m pip install --upgrade pip

# Install LibreLane ONLY (no openroad/klayout here)
!python -m pip install librelane

# Verify
!librelane --version



conda-forge/linux-64                                        Using cache
conda-forge/noarch                                          Using cache


Transaction

  Prefix: /content/conda-env

  Updating specs:

   - python=3.10
   - git
   - make


  Package                  Version  Build                 Channel           Size
──────────────────────────────────────────────────────────────────────────────────
  Install:
──────────────────────────────────────────────────────────────────────────────────

  + _libgcc_mutex              0.1  conda_forge           conda-forge     Cached
  + _openmp_mutex              4.5  2_gnu                 conda-forge     Cached
  + bzip2                    1.0.8  hda65f42_8            conda-forge     Cached
  + c-ares                  1.34.6  hb03c661_0            conda-forge      208kB
  + ca-certificates       2026.1.4  hbd8a1cb_0            conda-forge     Cached
  + git                     2.52.0  pl5321h6d3cee1_1      conda-forge       11MB
  + icu  

## Write verilog

In [10]:
%%writefile inverter.v
module inverter(input wire in, output wire out);
    assign out = !in;
endmodule

Overwriting inverter.v


## Write configuration

[Documentation](https://openlane.readthedocs.io/en/latest/reference/configuration.html)

In [11]:
%%writefile config.json
{
    "DESIGN_NAME": "inverter",
    "VERILOG_FILES": "dir::inverter.v",
    "CLOCK_TREE_SYNTH": false,
    "CLOCK_PORT": null,
    "PL_RANDOM_GLB_PLACEMENT": true,
    "FP_SIZING": "absolute",
    "DIE_AREA": "0 0 34.5 57.12",
    "PL_TARGET_DENSITY": 0.75,
    "FP_PDN_AUTO_ADJUST": false,
    "FP_PDN_VPITCH": 25,
    "FP_PDN_HPITCH": 25,
    "FP_PDN_VOFFSET": 5,
    "FP_PDN_HOFFSET": 5,
    "DIODE_INSERTION_STRATEGY": 3
}

Overwriting config.json


##  Run LibreLane Flow

[LibreLane](https://librelane.readthedocs.io/) is a modern, automated **RTL-to-GDSII** digital design flow and the actively maintained successor to OpenLane. It enables end-to-end physical implementation of digital designs, from **RTL** to **GDSII**, using a reproducible and Python-orchestrated methodology.

LibreLane is built on top of proven open-source EDA tools including **:contentReference[oaicite:0]{index=0}**, **:contentReference[oaicite:1]{index=1}**, **:contentReference[oaicite:2]{index=2}**, and **:contentReference[oaicite:3]{index=3}**. These tools are integrated through a unified flow that supports design space exploration, timing closure, power optimization, and signoff.

The LibreLane flow targets **open-source PDKs** (such as Sky130) and is well suited for learning, research, prototyping, and production-oriented open-silicon development.




In [12]:
%env PDK=sky130A
!flow.tcl -design .

env: PDK=sky130A
/bin/bash: line 1: flow.tcl: command not found


## Display layout

In [5]:
import pathlib
import gdstk
import IPython.display

gdss = sorted(pathlib.Path('runs').glob('*/results/final/gds/*.gds'))
library = gdstk.read_gds(gdss[-1])
top_cells = library.top_level()
top_cells[0].write_svg('inverter.svg')
IPython.display.SVG('inverter.svg')

IndexError: list index out of range

## Metrics

[Documentation](https://openlane.readthedocs.io/en/latest/reference/datapoint_definitions.html)


In [ ]:
import pandas as pd
import pathlib

pd.options.display.max_rows = None
reports = sorted(pathlib.Path('runs').glob('*/reports/metrics.csv'))
df = pd.read_csv(reports[-1])
df.transpose()